In [36]:
import pandas as pd
import numpy as np
np.int=int
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord
import astropy.units as u
import sfdmap

import os
import time
import extinction
from jedi.inference.compiled import create_from_access_path

In [37]:
CFA_PARAM_FILE = 'cfasnIa_param.dat'
OUTPUT_FILE = 'supernova_extinction_data.csv'

# Initialize SIMBAD and sfdmap
Simbad.add_votable_fields('ra', 'dec') # Request RA and Dec in degrees
sfd = sfdmap.SFDMap('/Users/pxm588@student.bham.ac.uk/Desktop/snid/all_raw_cfa_spectra/sfddata')


In [38]:
def get_sn_names(file_path):
    """
    Parses the cfasnIa_param.dat file to extract unique supernova names.
    """
    sn_names = set()
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue

            # SN name is the first column
            parts = line.split()
            if parts:
                sn_names.add(parts[0])
    return sorted(list(sn_names))

In [39]:
testname = get_sn_names(CFA_PARAM_FILE)[0]

In [40]:
print(testname)

1993ac


In [41]:
def query_simbad_and_sfd(sn_name):
    """
    Queries SIMBAD for supernova coordinates and then uses sfdmap for galactic extinction.
    """
    ra, dec, ebv = np.nan, np.nan, np.nan
    candidate_simbad_ids = []
    if sn_name.startswith('SNF'):
        candidate_simbad_ids.append(sn_name)
    else:
        candidate_simbad_ids.append('sn'+ sn_name)

    for simbad_id_to_try in candidate_simbad_ids:
        try:
            result_table = Simbad.query_object(simbad_id_to_try)
            if result_table is not None and len(result_table) > 0:
                ra_deg = result_table['ra'][0]
                dec_deg = result_table['dec'][0]

                coords = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame='icrs')
                ra, dec = coords.ra.deg, coords.dec.deg

                ebv = sfd.ebv(ra, dec, scale_by_r_v=False)

                print(f"Found {sn_name} using SIMBAD ID '{simbad_id_to_try}': RA={ra:.4f}, Dec={dec:.4f}, E(B-V)={ebv:.4f}")
                return ra, dec, ebv # Success!
            else:
                print(f"Could not find {sn_name} using SIMBAD ID '{simbad_id_to_try}'.")
        except Exception as e:
            print(f"Error querying SIMBAD for {sn_name} using SIMBAD ID '{simbad_id_to_try}': {e}")

    return ra, dec, ebv # If nothing found after all attempts


In [42]:
test_results = query_simbad_and_sfd(testname)

Found 1993ac using SIMBAD ID 'sn1993ac': RA=86.5982, Dec=63.3686, E(B-V)=0.1390


In [49]:
(print(test_results))
test_ebv = sfd.ebv(test_results[0], test_results[1])
print(test_ebv)

(86.59825, 63.36863888888889, 0.13901358988422918)
0.13901358988422918


In [44]:
test_spectra = 'all_spectra/sn1993ac-19931016.49-mmt.flm'

In [45]:
test_df = pd.read_csv(test_spectra, sep=r"\s+", header=None)
print(test_df.head())
# Extract columns for clarity
# Assumes: Col 0 = Wavelength, Col 1 = Flux, Col 2 = Error
wave = test_df.iloc[:, 0].values.astype(float)
flux = test_df.iloc[:, 1].values.astype(float)


         0             1    2
0  3150.00 -6.646612e-18  0.0
1  3151.95  1.829195e-17  0.0
2  3153.90  1.699877e-17  0.0
3  3155.84  2.819697e-17  0.0
4  3157.79 -2.667621e-17  0.0


In [46]:
# Dereddening logic
av = 3.1 * test_ebv
# Ensure wave is in Angstroms for fitzpatrick99
a_lambda = extinction.fitzpatrick99(wave, av, 3.1)
deredden_factor = 10**(0.4 * a_lambda)

In [47]:
test_df.iloc[:, 1] = flux * deredden_factor

# Fail-safe for Error Column
if test_df.shape[1] > 2:
    errors = pd.to_numeric(test_df.iloc[:, 2], errors='coerce')
    if not errors.isna().all():
        test_df.iloc[:, 2] = errors.values * deredden_factor
    else:
        print("Warning: Column 2 contains no valid numeric errors.")

In [48]:
print(test_df.head())

         0             1    2
0  3150.00 -1.314730e-17  0.0
1  3151.95  3.616909e-17  0.0
2  3153.90  3.359980e-17  0.0
3  3155.84  5.571401e-17  0.0
4  3157.79 -5.269003e-17  0.0
